In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

# 1. GPU 장치 설정
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(device)

mps


In [2]:

# 2. 데이터 전처리 (ResNet 표준 이미지 크기인 224x224로 변환)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # 데이터 증강
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet 기준 정규화
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet 기준 정규화
])


In [3]:

# 3. Food101 데이터셋 로드 (최초 실행 시 자동 다운로드)
train_dataset = torchvision.datasets.Food101(
    root="./data", split="train", download=True, transform=train_transform
)
test_dataset = torchvision.datasets.Food101(
    root="./data", split="test", download=True, transform=test_transform
)

In [4]:
model = resnet18(weights=None)

In [5]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [6]:
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=2
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=32, shuffle = False, num_workers=2
)

In [7]:

# 4. 사전 학습된 ResNet18 모델 불러오기
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# 5. 마지막 출력 레이어를 Food101 클래스 수(101개)로 변경
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 101)
model = model.to(device)

# 6. 손실함수 및 옵티마이저 설정
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"모델 준비 완료! 최종 출력 레이어 구조: {model.fc}")
# 이후 일반적인 train 루프를 돌리시면 됩니다.


모델 준비 완료! 최종 출력 레이어 구조: Linear(in_features=512, out_features=101, bias=True)


In [15]:
next(iter(train_loader))

[tensor([[[[-2.1008, -2.1179, -2.1179,  ...,  0.1939,  0.1597,  0.1939],
           [-2.1008, -2.1179, -2.1179,  ...,  0.1426,  0.2282,  0.4508],
           [-2.1179, -2.1008, -2.1008,  ...,  0.0741,  0.2453,  0.6563],
           ...,
           [-1.2103, -1.1932, -1.0390,  ..., -0.6965, -0.6452, -0.8164],
           [-1.0733, -1.1589, -1.1760,  ..., -0.7650, -0.5938, -0.6965],
           [-0.9020, -0.9534, -1.1932,  ..., -0.7479, -0.5938, -0.5767]],
 
          [[-1.4055, -1.4230, -1.4230,  ...,  0.0651, -0.1099, -0.1450],
           [-1.4055, -1.4230, -1.4405,  ...,  0.0476,  0.0301,  0.1877],
           [-1.4230, -1.4055, -1.4055,  ..., -0.0049,  0.1176,  0.5028],
           ...,
           [-1.3880, -1.4405, -1.3704,  ..., -1.2304, -1.1954, -1.3880],
           [-1.3704, -1.4405, -1.4230,  ..., -1.3179, -1.1779, -1.3529],
           [-1.3004, -1.2479, -1.3880,  ..., -1.3354, -1.2304, -1.2479]],
 
          [[ 0.2348,  0.2173,  0.1999,  ..., -0.3055, -0.4101, -0.3927],
           [ 

In [10]:
for i, (data, label) in enumerate(train_loader):
    print(i, data, label)

    if i ==0:
        break


0 tensor([[[[-1.2788, -1.2445, -1.2617,  ..., -0.7650, -0.6452, -0.6281],
          [-1.2617, -1.2788, -1.2788,  ..., -0.7308, -0.5938, -0.5938],
          [-1.2445, -1.2617, -1.2617,  ..., -0.7137, -0.6109, -0.6452],
          ...,
          [ 2.2147,  2.2147,  2.1290,  ..., -0.7308, -0.6452, -0.5082],
          [ 2.2318,  2.2489,  2.1804,  ..., -0.6965, -0.6794, -0.7137],
          [ 2.2489,  2.2489,  2.2147,  ..., -0.7137, -0.6965, -0.7137]],

         [[-1.1604, -1.1604, -1.1954,  ..., -1.1253, -1.0378, -1.0378],
          [-1.1429, -1.1604, -1.1779,  ..., -1.0728, -0.9678, -0.9853],
          [-1.1253, -1.1429, -1.1429,  ..., -1.0378, -0.9328, -1.0203],
          ...,
          [ 2.4111,  2.3936,  2.3060,  ..., -0.3025, -0.2675, -0.1625],
          [ 2.4111,  2.4286,  2.3585,  ..., -0.2675, -0.3025, -0.3901],
          [ 2.4286,  2.4286,  2.3936,  ..., -0.2850, -0.3025, -0.3901]],

         [[-1.2119, -1.0724, -0.9678,  ..., -1.5779, -1.6302, -1.6650],
          [-1.1944, -1.1247,

In [17]:
type(train_loader)

torch.utils.data.dataloader.DataLoader

In [ ]:
import torch
from tqdm.auto import tqdm

ep = 100

for epoch in range(ep):
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(
        train_loader,
        total=len(train_loader),
        desc=f"Epoch {epoch+1}/{ep} [Train]",
        ncols=100,
        leave=True,
    )

    for data, label in train_bar:
        data = data.to(device)
        label = label.to(device)

        optimizer.zero_grad()

        out = model(data)
        loss = criterion(out, label)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * data.size(0)

        pred = out.argmax(dim=1)
        train_correct += (pred == label).sum().item()
        train_total += label.size(0)

        train_bar.set_postfix(
            loss=f"{train_loss / train_total:.4f}",
            acc=f"{train_correct / train_total:.4f}",
        )

    train_loss /= train_total
    train_acc = train_correct / train_total

    model.eval()

    test_loss = 0.0
    test_correct = 0
    test_total = 0

    test_bar = tqdm(
        test_loader,
        total=len(test_loader),
        desc=f"Epoch {epoch+1}/{ep} [Test]",
        ncols=100,
        leave=True,
    )

    with torch.no_grad():
        for data, label in test_bar:
            data = data.to(device)
            label = label.to(device)

            out = model(data)
            loss = criterion(out, label)

            test_loss += loss.item() * data.size(0)

            pred = out.argmax(dim=1)
            test_correct += (pred == label).sum().item()
            test_total += label.size(0)

            test_bar.set_postfix(
                loss=f"{test_loss / test_total:.4f}",
                acc=f"{test_correct / test_total:.4f}",
            )

    test_loss /= test_total
    test_acc = test_correct / test_total

    tqdm.write(
        f"Epoch [{epoch+1}/{ep}] "
        f"train_loss: {train_loss:.4f}, "
        f"train_acc: {train_acc:.4f}, "
        f"test_loss: {test_loss:.4f}, "
        f"test_acc: {test_acc:.4f}"
    )

Epoch 1/100 [Train]:   9%|█▎            | 216/2368 [00:59<09:54,  3.62it/s, acc=0.1253, loss=3.8849]

In [ ]:
class Test:
    def __init__(self, num):
        self.num = num
    def __iter__(self):
        return IterTest(self.num)
    
class IterTest:
    def __init__(self, max):
        self.num = 0
        self.max = max
    def __next__(self):
        val = self.num
        self.num += 1
        if val == self.max:
            raise StopIteration
        return val
dd = iter(Test(3))
next(dd)
next(dd)
next(dd)
next(dd)




StopIteration: 

In [27]:
for i in Test(3):
    print(i)

0
1
2
